# Notebook 03: Dry-Run Evaluation

## Purpose

This notebook executes, **on Kaggle only**, a synthetic dry-run evaluation that
validates the full benchmark pipeline without any real OpenRouter call, without
any API key, without token spend, and without GPU.

It validates: dataset loading, schema validation, model pool loading, dry-run
runner, BudgetGuard, CostLedger, validators, per-model metrics, per-model task
breakdown, strict JSON serialization, leaderboard generation, and a
Hugging Face Space-compatible leaderboard artifact.

The dry-run uses synthetic responses derived from `example.expected`. These are
**pipeline validation only**, not model performance.


## Kaggle-only policy

- Execute **only on Kaggle**. Do not run locally.
- CPU only. No GPU.
- No OpenRouter call. No API key. No inference. No token spend. No training.
- No large model downloads.
- The project source is attached as a Kaggle dataset and imported via sys.path.
- `dry_run=True` is enforced and confirmed in outputs.


## 1. Environment setup and project location


In [ ]:
import sys
import platform
import subprocess
import shutil
import datetime
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
EXECUTED_AT = datetime.datetime.now(datetime.timezone.utc).isoformat()
print("Executed at (UTC):", EXECUTED_AT)

# Diagnosis: show what is mounted under /kaggle/input.
INPUT_BASE = Path("/kaggle/input")
print("\n/kaggle/input top-level entries:")
if INPUT_BASE.exists():
    for p in sorted(INPUT_BASE.iterdir()):
        print(" -", p)
else:
    print(" /kaggle/input does not exist")


## 2. Locate and copy the project source


In [ ]:
# Robustly find the project root by recursively scanning /kaggle/input for the
# github src marker. This handles any dataset mount nesting.
SRC_MARKER_REL = "work/5_final-work/github/src/slm_efficiency_frontier/__init__.py"

INPUT_PROJECT = None
matches = []
if INPUT_BASE.exists():
    matches = list(INPUT_BASE.rglob(SRC_MARKER_REL))
print("marker matches found:", len(matches))
for m in matches[:5]:
    print(" match:", m)

if matches:
    # match = <PROJECT>/work/5_final-work/github/src/slm_efficiency_frontier/__init__.py
    # parents[5] is the directory containing work/
    INPUT_PROJECT = matches[0].parents[5]
    print("Detected INPUT_PROJECT:", INPUT_PROJECT)

if INPUT_PROJECT is None:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / SRC_MARKER_REL).exists():
            INPUT_PROJECT = candidate
            break

print("INPUT_PROJECT:", INPUT_PROJECT)

WORK_COPY = Path("/kaggle/working/slm-efficiency-frontier")
if INPUT_PROJECT is not None:
    if WORK_COPY.exists():
        shutil.rmtree(WORK_COPY)
    shutil.copytree(INPUT_PROJECT, WORK_COPY)
    PROJECT_ROOT = WORK_COPY
else:
    PROJECT_ROOT = Path.cwd()

GITHUB_DIR = PROJECT_ROOT / "work" / "5_final-work" / "github"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("GITHUB_DIR:", GITHUB_DIR)
print("GITHUB_DIR exists:", GITHUB_DIR.exists())
print("src marker exists:", (GITHUB_DIR / "src" / "slm_efficiency_frontier" / "__init__.py").exists())


## 3. Import the project package


In [ ]:
# Import via sys.path (robust; does not require editable install).
SRC_DIR = GITHUB_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Ensure runtime deps are present (torch is preinstalled on Kaggle).
for dep in ("pyyaml", "requests"):
    try:
        __import__(dep)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", dep])

import slm_efficiency_frontier
print("slm_efficiency_frontier imported, version:", slm_efficiency_frontier.__version__)

import json
import dataclasses
import importlib.util
from slm_efficiency_frontier.config import load_config
from slm_efficiency_frontier.budget import BudgetGuard, CostLedger
from slm_efficiency_frontier.datasets import BenchmarkDataset
from slm_efficiency_frontier.evaluator import Evaluator
from slm_efficiency_frontier.openrouter import OpenRouterRunner, DRY_RUN_BACKEND
from slm_efficiency_frontier.validators import VALIDATORS
from slm_efficiency_frontier.json_utils import sanitize_for_json, dump_json
print("All project modules imported successfully.")


## 4. Load config, model pool, and sample dataset


In [ ]:
CONFIGS_DIR = GITHUB_DIR / "configs"
DATA_PATH = GITHUB_DIR / "data" / "sample" / "examples.jsonl"

config = load_config(CONFIGS_DIR)
print("Budget config:", config.budget)
print("Models in pool:", len(config.models))

dataset = BenchmarkDataset(DATA_PATH)
print("Sample dataset size:", len(dataset))
print("Task families:", sorted({ex.task for ex in dataset.examples}))


## 5. Select dry-run candidates


In [ ]:
ledger = CostLedger()
guard = BudgetGuard(config.budget, ledger, config.models)

eligible_models = [m for m in config.models if guard.is_eligible(m.model_id)]
free_models = [m for m in eligible_models if ":free" in m.model_id]
ineligible_models = [m for m in config.models if not guard.is_eligible(m.model_id)]

print("Eligible in pool:", len(eligible_models))
print("Free eligible:", len(free_models))
print("Ineligible in pool:", len(ineligible_models))

candidates = []
DEEPSEEK = "deepseek/deepseek-v4-flash"
if DEEPSEEK in {m.model_id for m in eligible_models}:
    candidates.append(DEEPSEEK)

cheap = sorted(
    [m for m in eligible_models if m.model_id not in candidates and ":free" not in m.model_id],
    key=lambda m: (m.input_price_per_million or 0.0) + (m.output_price_per_million or 0.0),
)
for m in cheap[:5]:
    if m.model_id not in candidates:
        candidates.append(m.model_id)

for m in free_models:
    if m.model_id not in candidates:
        candidates.append(m.model_id)
        break

for m in ineligible_models:
    if m.model_id not in candidates:
        candidates.append(m.model_id)
        break

candidates = candidates[:8]
print("Dry-run candidates (max 8):")
for c in candidates:
    print(" -", c, "| eligible:", guard.is_eligible(c))


## 6. Run dry-run evaluation


In [ ]:
runner = OpenRouterRunner(api_key=None, budget_guard=guard, ledger=ledger, dry_run=True)
evaluator = Evaluator(runner, VALIDATORS, seed=42)

assert runner.dry_run is True, "dry_run must be True"
report = evaluator.evaluate(candidates, dataset.examples)

print("dry_run:", report.dry_run)
print("models in report:", report.models)
print("total_cost_usd:", report.total_cost_usd)
print("per_model_metrics keys:", list(report.per_model_metrics.keys()))
print("ledger entries:", len(ledger.entries))
print("remaining budget USD:", guard.remaining_usd())


## 7. Verify model identity preservation


In [ ]:
identity_evidence = []
sample_ex = dataset.examples[0]
for mid in candidates:
    r = runner.run_example(mid, sample_ex)
    identity_evidence.append({
        "candidate_model_id": mid,
        "runresult_model_id": r.model_id,
        "execution_model_id": r.execution_model_id,
        "eligible": guard.is_eligible(mid),
        "used_synthetic_backend": r.execution_model_id == DRY_RUN_BACKEND,
    })

for ev in identity_evidence:
    print(ev)

assert all(ev["runresult_model_id"] == ev["candidate_model_id"] for ev in identity_evidence)
for ev in identity_evidence:
    if not ev["eligible"]:
        assert ev["used_synthetic_backend"] is True, f"Ineligible {ev['candidate_model_id']} did not use synthetic backend"
print("Identity preservation verified.")


## 8. Build leaderboard


In [ ]:
BUILD_SCRIPT = GITHUB_DIR / "scripts" / "build_leaderboard.py"
spec = importlib.util.spec_from_file_location("build_leaderboard", BUILD_SCRIPT)
build_mod = importlib.util.module_from_spec(spec)
sys.modules["build_leaderboard"] = build_mod
spec.loader.exec_module(build_mod)

report_dict = dataclasses.asdict(report)
leaderboard = build_mod.build_leaderboard(report_dict)

print("Leaderboard rows:", len(leaderboard))
assert len(leaderboard) == len(candidates), "Leaderboard must have one row per candidate"
for row in leaderboard:
    print(row["model_id"], "| cost/correct:", row["cost_per_correct_answer"], "| acc:", row["accuracy"])


## 9. Strict JSON serialization and validation


In [ ]:
EVAL_ARTIFACTS_DIR = PROJECT_ROOT / "work" / "2_development"
REVIEW_DIR = PROJECT_ROOT / "work" / "4_for-review"
RELEASE_DIR = PROJECT_ROOT / "artifacts/release"
for d in (EVAL_ARTIFACTS_DIR, REVIEW_DIR, RELEASE_DIR):
    d.mkdir(parents=True, exist_ok=True)

REPORT_JSON = EVAL_ARTIFACTS_DIR / "dry_run_evaluation_report.json"
LEADERBOARD_JSON = EVAL_ARTIFACTS_DIR / "dry_run_leaderboard.json"
SUMMARY_MD = EVAL_ARTIFACTS_DIR / "dry_run_evaluation_summary.md"

dump_json(report_dict, REPORT_JSON, indent=2)
dump_json(leaderboard, LEADERBOARD_JSON, indent=2)

def validate_strict_json(path):
    raw = Path(path).read_text(encoding="utf-8")
    json.loads(raw)
    json.dumps(json.loads(raw), allow_nan=False)
    for token in ("Infinity", "-Infinity", "NaN"):
        assert token not in raw, f"{token} found in {path}"
    return True

report_ok = validate_strict_json(REPORT_JSON)
leaderboard_ok = validate_strict_json(LEADERBOARD_JSON)
print("Report strict JSON valid:", report_ok)
print("Leaderboard strict JSON valid:", leaderboard_ok)

sanitized_report = sanitize_for_json(report_dict)
sanitized_leaderboard = sanitize_for_json(leaderboard)
json.dumps(sanitized_report, allow_nan=False)
json.dumps(sanitized_leaderboard, allow_nan=False)
print("In-memory sanitized structures are strict-JSON safe.")


## 10. Generate dry-run summary markdown


In [ ]:
def build_summary():
    L = []
    L.append("# Dry-Run Evaluation Summary")
    L.append("")
    L.append(f"**Execution timestamp (UTC):** {EXECUTED_AT}")
    L.append("")
    L.append("## Kaggle runtime confirmation")
    L.append("")
    L.append("- Executed on Kaggle: yes")
    L.append("- CPU only: yes")
    L.append("- GPU used: no")
    L.append("")
    L.append("## No OpenRouter call confirmation")
    L.append("")
    L.append("- OpenRouter called: no")
    L.append("- API key used: no")
    L.append("- Token spend: 0 (synthetic dry-run)")
    L.append(f"- dry_run flag: {report.dry_run}")
    L.append("")
    L.append("## Model candidates used")
    L.append("")
    for c in candidates:
        L.append(f"- {c} (eligible: {guard.is_eligible(c)})")
    L.append("")
    L.append("## Sample dataset size")
    L.append("")
    L.append(f"- Examples: {len(dataset)}")
    L.append("")
    L.append("## Task families covered")
    L.append("")
    for task in sorted({ex.task for ex in dataset.examples}):
        L.append(f"- {task}")
    L.append("")
    L.append("## Result count")
    L.append("")
    L.append(f"- Candidates x examples = {len(candidates)} x {len(dataset)} = {len(candidates) * len(dataset)}")
    L.append("")
    L.append("## Per-model leaderboard summary")
    L.append("")
    L.append("| model_id | accuracy | cost_per_correct_answer | cost_per_1k | median_latency_ms |")
    L.append("|----------|----------|-------------------------|-------------|-------------------|")
    for row in leaderboard:
        L.append(f'| {row["model_id"]} | {row["accuracy"]} | {row["cost_per_correct_answer"]} | {row["cost_per_1k_examples"]} | {row["median_latency_ms"]} |')
    L.append("")
    L.append("## Per-model task breakdown summary")
    L.append("")
    for model_id, tasks in report.per_model_task_breakdown.items():
        L.append(f"### {model_id}")
        for task, metrics in tasks.items():
            L.append(f'- {task}: accuracy={metrics["accuracy"]}, cost/correct={metrics["cost_per_correct_answer"]}, count={metrics["count"]}')
        L.append("")
    L.append("## Strict JSON validation result")
    L.append("")
    L.append(f"- Report strict JSON valid: {report_ok}")
    L.append(f"- Leaderboard strict JSON valid: {leaderboard_ok}")
    L.append("- No Infinity/-Infinity/NaN tokens in artifacts.")
    L.append("")
    L.append("## BudgetGuard / CostLedger behavior")
    L.append("")
    L.append(f"- Ledger entries: {len(ledger.entries)}")
    L.append(f"- Total estimated cost (USD): {ledger.total}")
    L.append(f"- Remaining budget (USD): {guard.remaining_usd()}")
    L.append("- No real spend occurred (synthetic dry-run).")
    L.append("")
    L.append("## Model identity preservation")
    L.append("")
    L.append("| candidate | runresult_model_id | execution_model_id | synthetic_backend |")
    L.append("|-----------|--------------------|--------------------|-------------------|")
    for ev in identity_evidence:
        L.append(f'| {ev["candidate_model_id"]} | {ev["runresult_model_id"]} | {ev["execution_model_id"]} | {ev["used_synthetic_backend"]} |')
    L.append("")
    L.append("## Whether dry-run pipeline is ready")
    L.append("")
    L.append("Yes. The full pipeline (load -> dry-run -> validate -> metrics -> leaderboard -> strict JSON) ran end-to-end.")
    L.append("")
    L.append("## Limitations")
    L.append("")
    L.append("- Synthetic responses are derived from expected answers; accuracy is NOT model performance.")
    L.append("- Costs for eligible models are price-based estimates, not real spend.")
    L.append("- Real OpenRouter execution is still pending (_real_call, notebook 04).")
    L.append("")
    L.append("## Next step recommendation")
    L.append("")
    L.append("- Proceed to notebook 04 only after implementing _real_call(); until then do not run real inference.")
    L.append("- Run notebook 05 for final QA and determinism tests.")
    L.append("")
    return "\n".join(L)


summary_text = build_summary()
SUMMARY_MD.write_text(summary_text, encoding="utf-8")
print("Summary written to", SUMMARY_MD)


## 11. Update Review snapshots


In [ ]:
ARCH_SNAP = REVIEW_DIR / "architecture_snapshot.md"
QA_SNAP = REVIEW_DIR / "qa_snapshot.md"
RELEASE_SNAP = REVIEW_DIR / "release_snapshot.md"

ARCH_SNAP.write_text(
    "# Architecture Snapshot\n\n"
    f"**Updated by Kaggle notebook 03 at (UTC):** {EXECUTED_AT}\n\n"
    "## Pipeline validated (dry-run)\n"
    "- Dataset loading (torch Dataset) and schema validation.\n"
    "- Model pool loading and BudgetGuard eligibility.\n"
    "- OpenRouterRunner dry-run with synthetic backend preserving candidate model_id.\n"
    "- CostLedger records per-call estimates; no real spend.\n"
    "- Validators score synthetic responses.\n"
    "- Per-model metrics and per-model task breakdown.\n"
    "- Leaderboard: one row per model, ranked by cost_per_correct_answer.\n"
    "- Strict JSON serialization (no Infinity/NaN).\n\n"
    "## PyTorch usages\n"
    "- Dataset loaders subclass torch.utils.data.Dataset.\n"
    "- Metrics aggregate with torch.tensor.\n"
    "- Baselines are torch.nn.Module subclasses.\n\n"
    "## Real execution status\n"
    "- _real_call() is still NotImplementedError; pending notebook 04.\n",
    encoding="utf-8",
)

QA_SNAP.write_text(
    "# QA Snapshot\n\n"
    f"**Updated by Kaggle notebook 03 at (UTC):** {EXECUTED_AT}\n\n"
    "## Resolved\n"
    "- Q-001: HF duplication sweep (notebook 01).\n"
    "- Q-002: OpenRouter price verification (notebook 02).\n"
    "- Q-022: Dry-run pipeline validated end-to-end on Kaggle (notebook 03).\n\n"
    "## Open\n"
    "- Q-003: Final determinism tests (notebook 05).\n"
    "- Q-015 / K-001: Real OpenRouter execution (_real_call, notebook 04).\n\n"
    "## Legitimacy QA\n"
    "- Total: 85/100 ACCEPTED (from notebook 02).\n",
    encoding="utf-8",
)

RELEASE_SNAP.write_text(
    "# Release Snapshot\n\n"
    f"**Updated by Kaggle notebook 03 at (UTC):** {EXECUTED_AT}\n\n"
    "## Notebook status\n"
    "- Notebook 01: completed on Kaggle (HF duplication sweep).\n"
    "- Notebook 02: completed on Kaggle (OpenRouter price verification; 21 eligible).\n"
    "- Notebook 03: completed on Kaggle (dry-run pipeline validation).\n"
    "- Notebook 04: NOT READY, pending _real_call() implementation.\n"
    "- Notebook 05: pending final QA.\n\n"
    "## Dry-run artifacts\n"
    "- artifacts/evaluation/dry_run_evaluation_report.json\n"
    "- artifacts/evaluation/dry_run_leaderboard.json\n"
    "- artifacts/evaluation/dry_run_evaluation_summary.md\n\n"
    "## v0.1 scope\n"
    "Strictly English-only. No multilingual content in final artifacts.\n\n"
    "## Not yet done\n"
    "- Real evaluation run (notebook 04).\n"
    "- Push/publish (awaiting user confirmation).\n",
    encoding="utf-8",
)
print("Review snapshots updated.")


## 12. Update memory and QA


In [ ]:
QA_FINDINGS_PATH = RELEASE_DIR / "qa_findings.md"
CHANGE_LOG_PATH = RELEASE_DIR / "change_log.md"
DECISIONS_PATH = RELEASE_DIR / "decisions.md"

qa = []
qa.append("# QA Findings")
qa.append("")
qa.append("| ID | Area | Severity | Description | Status |")
qa.append("|----|------|----------|-------------|--------|")
qa.append("| Q-001 | research | BLOCKER | HF duplication sweep executed on Kaggle; max risk 45; 0 duplicates. | resolved |")
qa.append("| Q-002 | budget | MAJOR | OpenRouter price verification executed on Kaggle; 21 eligible models. | resolved |")
qa.append("| Q-003 | metrics | MINOR | Determinism tests not yet run (notebook 05). | open |")
qa.append(f"| Q-022 | dry-run | MAJOR | Dry-run pipeline validated end-to-end on Kaggle at {EXECUTED_AT}; strict JSON, per-model leaderboard, identity preservation confirmed. | resolved |")
qa.append("| Q-015 | runtime | MAJOR | Real OpenRouter execution (_real_call) not implemented. | open |")
QA_FINDINGS_PATH.write_text("\n".join(qa) + "\n", encoding="utf-8")

entry = (
    f"\n## {EXECUTED_AT} (Kaggle notebook 03 executed)\n"
    "- Executed dry-run evaluation on Kaggle (CPU, no OpenRouter, no API key, no spend).\n"
    f"- Candidates: {len(candidates)}; examples: {len(dataset)}; "
    f"results: {len(candidates) * len(dataset)}.\n"
    f"- Leaderboard rows: {len(leaderboard)} (one per model).\n"
    f"- Strict JSON valid (no Infinity/NaN). Ledger entries: {len(ledger.entries)}.\n"
    "- Generated dry_run_evaluation_report.json, dry_run_leaderboard.json, "
    "dry_run_evaluation_summary.md, architecture_snapshot.md, qa_snapshot.md, "
    "release_snapshot.md.\n"
    "- Marked Q-022 resolved; Q-003 and Q-015 remain open.\n"
)
with open(CHANGE_LOG_PATH, "a", encoding="utf-8") as fh:
    fh.write(entry)

dr = (
    f"\n## DR-014: Dry-run pipeline validated ({EXECUTED_AT}, Kaggle)\n"
    "- **Decision:** The dry-run pipeline is ready; proceed to prepare notebook 04.\n"
    "- **Rationale:** Notebook 03 validated dataset loading, schema, model pool, "
    "BudgetGuard, CostLedger, dry-run runner with identity preservation, validators, "
    "per-model metrics, per-model task breakdown, leaderboard (one row per model), "
    "and strict JSON serialization on Kaggle.\n"
    "- **Consequences:** Q-022 resolved. Real execution still pending _real_call().\n"
    "- **Next action:** Implement _real_call() in notebook 04; run notebook 05 for final QA.\n"
)
with open(DECISIONS_PATH, "a", encoding="utf-8") as fh:
    fh.write(dr)

print("Memory and QA updated.")


## Next steps

1. Inspect `artifacts/evaluation/dry_run_evaluation_summary.md`.
2. Do NOT interpret dry-run accuracy as model performance.
3. Implement `_real_call()` in notebook 04 before any real inference.
4. Run notebook 05 for final QA and determinism tests.

This notebook did not call OpenRouter, did not use an API key, did not spend
tokens, did not use GPU, and did not download models. It is synthetic dry-run
pipeline validation only.
